In [0]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from pyspark.sql.types import StructType , StructField , StringType, IntegerType, DateType , DoubleType, TimestampType

In [0]:
# %sql
# create or replace table intellibi_catalog.intellibi_bronze.rawMonthlySales (
# sale_id integer,
# product string,
# category string,
# quantity integer,
# price double,
# sale_date date,
# region string,
# ingest_ts timestamp
# );

# -- note that here we can make all data type string too so that no error will be there while reading ghe data using csv file. then in next layer silver, we can do all the data type conversion.

In [0]:
# Schema_sales = StructType(
# [
# StructField('sale_id', IntegerType(), True),
# StructField('product', StringType(), True),
# StructField('category', StringType(), True),
# StructField('quantity', IntegerType(), True),
# StructField('price',  DoubleType(), True),
# StructField('sale_date', DateType(),True),
# StructField('region', StringType(), True),
# StructField('ingest_ts', TimestampType(), True) 

# ])  


In [0]:
%sql
-- Incremental load is called as "Watermark Table" or "CDC"/'Change data capture'. Hence here rawMonthlySales will be considered as wateramark table. the incremental load is done using copy into command

copy into intellibi_catalog.intellibi_bronze.rawMonthlySales
from (
select cast(sale_id as int) as sale_id ,
product ,
category ,
cast(quantity as int) as quantity ,
cast(price as double) as price ,
cast(sale_date as date) as sale_date ,
region, 
current_timestamp() as ingest_ts 
from 'dbfs:/Volumes/intellibi_catalog/intellibi_schema/intellibi_data_engineering_volume/src_files/monthly_sales/' )
FILEFORMAT = CSV
FORMAT_OPTIONS ('header'='true')
COPY_OPTIONS ('mergeSchema'='false')

In [0]:
# %sql
# select * from intellibi_catalog.intellibi_bronze.rawMonthlySales

In [0]:
# %sql
# DESCRIBE HISTORY intellibi_catalog.intellibi_bronze.rawMonthlySales
# -- using this history command we can seee all the operatin happend in past in this table 

In [0]:

# in real time the data is not thetre in VOLUME , actually it is there in AMAZON S3 location , just for practice we are doing this




In [0]:
# now i need to lolad this clean data to silver table hence for that we need to create a silver schema and a silver table which is emplty table 



In [0]:
# %sql
# INSERT INTO intellibi_catalog.intellibi_bronze.rawMonthlySales
# SELECT 
#     sale_id,
#     product,
#     category,
#     quantity + 10 AS quantity,       -- Changed value to distinguish the newer duplicate
#     price,
#     sale_date,
#     region,
#     current_timestamp() AS ingest_ts -- Newer timestamp so QUALIFY knows which row is latest
# FROM intellibi_catalog.intellibi_bronze.rawMonthlySales
# WHERE sale_id = 1
# LIMIT 1;


In [0]:
# %sql
# SELECT sale_id, product, quantity, price, ingest_ts
# FROM intellibi_catalog.intellibi_bronze.rawMonthlySales
# WHERE sale_id = 1
# ORDER BY ingest_ts DESC;
